In [4]:
%load_ext autoreload
%autoreload 2

import os
import sys
import requests
import pandas as pd
import time
import datetime
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
load_dotenv("../../config/.env")
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")

notebook_dir = os.path.dirname(os.path.abspath("__file__"))
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

raw_ingest_file = os.path.join(raw_data_dir, "raw_hl7.csv")

# Import Schema Manager
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry Lookup & Target Setup ---
# We define the targets, but ask the registry for the official metadata
HL7_TARGETS = {
    "HL7v3": {"url": "http://terminology.hl7.org/CodeSystem-v3-ReligiousAffiliation", "name": "HL7 v3", "base": "http://terminology.hl7.org/CodeSystem/v3-ReligiousAffiliation"},
    "HL7v2": {"url": "http://terminology.hl7.org/CodeSystem-v2-0006", "name": "HL7 v2", "base": "http://terminology.hl7.org/CodeSystem/v2-0006"}
}

HEADERS = {
    'User-Agent': f'ReligiousMappingProject/1.0 (Research; mailto:{contact_email})',
    'Accept': 'application/fhir+json'
}

# --- 3. Extraction Logic ---
def process_hl7_item(item, prefix, base_uri, source_name, parent_id=None, parent_path=""):
    """
    Recursively parses FHIR CodeSystem concepts. 
    Passes data through finalize_row() for 14-column alignment.
    """
    rows = []
    
    # Core Identifiers
    code = str(item.get('code', ''))
    display = item.get('display', 'Unknown')
    system = item.get('system', base_uri)
    
    if not code: return []

    # Build Hierarchy Path
    current_path = f"{parent_path} > {display}" if parent_path else display
    
    # Extraction of Raw Definition
    description_value = item.get('definition', "")

    # --- UPGRADED: Dynamic Synonym & Translation Extraction ---
    synonym_list = []
    has_translation = ""
    
    for d in item.get('designation', []):
        val = d.get('value')
        lang = d.get('language', '').lower()
        
        if val:
            # If no language tag exists, or it starts with 'en', treat it as an English synonym
            if not lang or lang.startswith('en'):
                if val != display:
                    synonym_list.append(val)
            else:
                # If any non-English language tag appears, flag it for the future!
                has_translation = "yes"

    # Capture Status (Deprecation)
    concept_status = "active"
    for prop in item.get('property', []):
        p_code = prop.get('code')
        if p_code == 'status' and prop.get('valueCode') == 'deprecated':
            concept_status = "deprecated"
        elif p_code == 'deprecationDate':
            dt = prop.get('valueDateTime', 'unknown date')
            concept_status = f"deprecated ({dt})"

    # Pack the raw data
    extracted_data = {
        'Source_System': source_name,
        'Primary_Label': display,
        'CURIE': f"{prefix}:{code}",
        'Formal_Label': display,
        'Concept_Type': 'skos:Concept',
        'Hierarchy_Path': current_path,
        'Synonyms': " | ".join(synonym_list),
        'Description': description_value,
        'Parent_IDs': parent_id if parent_id else "",
        'Concept_ID': code,
        'URI': f"{system}#{code}",
        'Has_Translation': has_translation, # Now dynamically evaluated!
        'Status': concept_status
    }
    
    # MAGIC HAPPENS HERE: Force data into the strict 14-column schema
    clean_row = finalize_row(extracted_data)
    rows.append(clean_row)

    # Recursive Walk for nested children
    for sub_item in item.get('concept', []):
        rows.extend(process_hl7_item(sub_item, prefix, base_uri, source_name, code, current_path))
            
    return rows

# --- 4. Main Execution ---
if os.path.exists(raw_ingest_file): os.remove(raw_ingest_file)

for target_prefix, info in HL7_TARGETS.items():
    
    # Validate against registry, auto-stubbing if missing
    registry_data = get_registry_info(
        prefix=target_prefix, 
        config_dir=config_dir, 
        fallback_name=info['name'],
        fallback_uri=info['base']
    )
    
    SOURCE_NAME = registry_data['Source_Name']
    BASE_URI = registry_data['Base_URI']
    
    print(f"\n--- Harvesting {SOURCE_NAME} ({target_prefix}) ---")
    
    url = f"{info['url']}.json"
    success = False
    
    # Retry logic
    for attempt in range(3):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.raise_for_status()
            data = response.json()
            success = True
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(2)
            
    if not success:
        print(f"Failed to harvest {target_prefix} after 3 attempts. Skipping.")
        continue
        
    all_rows = []
    root_concepts = data.get('concept', [])
    
    for item in root_concepts:
        all_rows.extend(process_hl7_item(item, target_prefix, BASE_URI, SOURCE_NAME))
        
    if not all_rows:
        print(f"Warning: No concepts found for {target_prefix}.")
        continue

    print(f"Captured {len(all_rows)} concepts.")
    
    # Save to raw_ingest_file with standard COLUMN_ORDER
    df = pd.DataFrame(all_rows)[COLUMN_ORDER]
    file_exists = os.path.isfile(raw_ingest_file)
    df.to_csv(raw_ingest_file, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')

print(f"\nDone! Objective HL7 data appended to {raw_ingest_file}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

--- Harvesting HL7 v3 (HL7v3) ---
Captured 82 concepts.

--- Harvesting HL7 v2 (HL7v2) ---
Captured 95 concepts.

Done! Objective HL7 data appended to c:\Users\njsgi\Documents\GitHub\religion-ontology-mapping\data\raw\raw_hl7.csv
